# Inception Variants Experiment

Motor imagery classification on the PhysioNet EEG Motor Movement/Imagery
Dataset. Trains and evaluates EEGNet baseline plus five inception-style
temporal variants, then produces a full cross-model comparison.

## 1. Imports & Config

In [ ]:
import os
import random
import warnings

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mne
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)

warnings.filterwarnings('ignore', message='Limited .* annotation.* outside the data range')
mne.set_log_level('WARNING')

# ── Reproducibility ───────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ['PYTHONHASHSEED'] = str(SEED)

# ── Paths & dataset constants ───────────────────────────────────────
DATA_DIR = os.path.join(os.getcwd(), 'physionet.org/files/eegmmidb/1.0.0')
RESULTS_DIR = os.path.join(os.getcwd(), 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

RUNS = ['R03', 'R04', 'R07', 'R08', 'R11', 'R12']
N_SUBJECTS = 109

SFREQ      = 160
TMIN       = 0.0
TMAX       = 2.0
N_SAMPLES  = int(SFREQ * (TMAX - TMIN))
N_CHANNELS = 64

NOTCH_FREQ = 60.0
BP_LOW     = 4.0
BP_HIGH    = 40.0

TRAIN_FRAC = 0.70
VAL_FRAC   = 0.15

BATCH_SIZE = 64
EPOCHS     = 50
LR         = 1e-3

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## 2. Data Loading & Preprocessing

In [ ]:
def load_subject(subject_id):
    """Load all runs for one subject and return (X, y).

    Returns
    -------
    X : ndarray, shape (n_epochs, n_channels, n_samples)
    y : ndarray, shape (n_epochs,)  — 0=left hand, 1=right hand
    """
    subject = f'S{subject_id:03d}'
    raws = []

    for run in RUNS:
        path = os.path.join(DATA_DIR, subject, f'{subject}{run}.edf')
        if not os.path.exists(path):
            continue
        raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
        # Notch / bandpass optional — disabled to match Inception.ipynb baseline
        # raw.notch_filter(NOTCH_FREQ, verbose=False)
        # raw.filter(BP_LOW, BP_HIGH, verbose=False)
        raws.append(raw)

    if not raws:
        return np.empty((0, N_CHANNELS, 0)), np.empty((0,), dtype=int)

    raw = mne.concatenate_raws(raws)
    events, event_id = mne.events_from_annotations(raw)

    if 'T1' not in event_id or 'T2' not in event_id:
        return np.empty((0, N_CHANNELS, 0)), np.empty((0,), dtype=int)

    epochs = mne.Epochs(
        raw, events,
        event_id={'T1': event_id['T1'], 'T2': event_id['T2']},
        tmin=TMIN, tmax=TMAX,
        baseline=None, preload=True, verbose=False,
    )

    X = epochs.get_data()
    y = epochs.events[:, -1]
    y = (y == event_id['T2']).astype(int)

    X_fixed = []
    for x in X:
        if x.shape[1] >= N_SAMPLES:
            x = x[:, :N_SAMPLES]
        else:
            x = np.pad(x, ((0, 0), (0, N_SAMPLES - x.shape[1])))
        X_fixed.append(x)

    return np.stack(X_fixed), y


def load_all_subjects(n_subjects=N_SUBJECTS):
    X_all, y_all = [], []
    for sid in range(1, n_subjects + 1):
        X, y = load_subject(sid)
        if len(X) == 0:
            continue
        X_all.append(X)
        y_all.append(y)
    return np.concatenate(X_all), np.concatenate(y_all)

In [ ]:
print('Loading all subjects...')
X_all, y_all = load_all_subjects()
print(f'Total epochs: {len(X_all)} | Shape: {X_all.shape}')
print(f'Class balance — left: {(y_all==0).sum()} | right: {(y_all==1).sum()}')

## 3. Normalisation & Random Train / Val / Test Split

In [ ]:
def normalize(X):
    """Per-epoch, per-channel z-score + add Conv2d channel dim."""
    mu  = X.mean(axis=2, keepdims=True)
    std = X.std(axis=2,  keepdims=True) + 1e-6
    X   = (X - mu) / std
    return X[:, np.newaxis, :, :]


class EEGDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
X_norm = normalize(X_all)

n_total = len(X_norm)
rng = np.random.default_rng(SEED)
idx = rng.permutation(n_total)

n_train = int(n_total * TRAIN_FRAC)
n_val   = int(n_total * VAL_FRAC)

train_idx = idx[:n_train]
val_idx   = idx[n_train:n_train + n_val]
test_idx  = idx[n_train + n_val:]

full_dataset = EEGDataset(X_norm, y_all)

train_loader = DataLoader(Subset(full_dataset, train_idx), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(Subset(full_dataset, val_idx),   batch_size=BATCH_SIZE)
test_loader  = DataLoader(Subset(full_dataset, test_idx),  batch_size=BATCH_SIZE)

print(f'Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}')

## 4. EEGNet Baseline

In [ ]:
class EEGNet(nn.Module):
    """Baseline EEGNet (Lawhern et al., 2018) with single temporal kernel of 64."""
    def __init__(self, n_channels=64, n_samples=320, n_classes=2,
                 F1=8, D=2, F2=16, dropout=0.25):
        super().__init__()

        # Block 1 — temporal convolution
        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, (1, 64), padding=(0, 32), bias=False),
            nn.BatchNorm2d(F1),
        )

        # Block 2 — depthwise spatial convolution
        self.block2 = nn.Sequential(
            nn.Conv2d(F1, F1 * D, (n_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout),
        )

        # Block 3 — separable convolution
        self.block3 = nn.Sequential(
            nn.Conv2d(F1 * D, F2, (1, 16), padding=(0, 8), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(dropout),
        )

        flatten_dim = F2 * (n_samples // 32)
        self.classifier = nn.Linear(flatten_dim, n_classes)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = x.flatten(start_dim=1)
        return self.classifier(x)

## 5. Inception Variants

Five architectures that replace EEGNet's single fixed temporal kernel with
parallel inception branches. Everything else (spatial conv, separable conv,
classifier) is identical to baseline EEGNet so results are directly comparable.

| Model | Branches | Kernel sizes (samples) | Notes |
|---|---|---|---|
| `EEGNet` | 1 | 64 | Baseline — half sampling rate |
| `EEGNet_Inc2_SmallMed` | 2 | 16, 32 | Higher-frequency bias |
| `EEGNet_Inc2_MedLarge` | 2 | 32, 64 | Mid/low-frequency bias |
| `EEGNet_Inc3` | 3 | 16, 32, 64 | Full-range, mirrors EEG-ITNet |
| `EEGNet_Inc3_Narrow` | 3 | 8, 16, 32 | Alpha/beta focused |
| `EEGNet_Inc3_Wide` | 3 | 32, 64, 128 | Low-frequency focused |

**Filter budget**: each branch gets `F1 // n_branches` filters so the total
entering the spatial conv stays fixed at `F1` across all variants.

In [ ]:
class InceptionTemporalBlock(nn.Module):
    """Drop-in replacement for EEGNet's single temporal Conv2d.

    Runs N parallel temporal convolutions with different kernel sizes,
    concatenates their outputs along the filter dimension, then projects
    back to F1 filters so the spatial block sees the same tensor shape
    regardless of how many branches were used.
    """
    def __init__(self, kernel_sizes, F1=8):
        super().__init__()
        n = len(kernel_sizes)
        base = F1 // n
        branch_filters = [base] * n
        branch_filters[-1] += F1 - base * n  # absorb remainder

        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(1, f, (1, k), padding=(0, k // 2), bias=False),
                nn.BatchNorm2d(f),
            )
            for k, f in zip(kernel_sizes, branch_filters)
        ])

        total_branch_filters = sum(branch_filters)
        self.project = (
            nn.Conv2d(total_branch_filters, F1, (1, 1), bias=False)
            if total_branch_filters != F1
            else nn.Identity()
        )
        self.bn_out = nn.BatchNorm2d(F1)

    def forward(self, x):
        outs = [branch(x) for branch in self.branches]
        min_t = min(o.shape[-1] for o in outs)
        outs = [o[..., :min_t] for o in outs]
        x = torch.cat(outs, dim=1)
        x = self.project(x)
        return self.bn_out(x)

In [ ]:
class EEGNetInception(nn.Module):
    """EEGNet with block1 replaced by an InceptionTemporalBlock."""
    def __init__(self, n_channels=64, n_samples=320, n_classes=2,
                 F1=8, D=2, F2=16, dropout=0.25,
                 kernel_sizes=(16, 32, 64)):
        super().__init__()

        # Block 1 — inception temporal convolution
        self.block1 = InceptionTemporalBlock(kernel_sizes=kernel_sizes, F1=F1)

        # Block 2 — depthwise spatial convolution (unchanged)
        self.block2 = nn.Sequential(
            nn.Conv2d(F1, F1 * D, (n_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout),
        )

        # Block 3 — separable convolution (unchanged)
        self.block3 = nn.Sequential(
            nn.Conv2d(F1 * D, F2, (1, 16), padding=(0, 8), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(dropout),
        )

        flatten_dim = F2 * (n_samples // 32)
        self.classifier = nn.Linear(flatten_dim, n_classes)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = x.flatten(start_dim=1)
        return self.classifier(x)

In [ ]:
COMMON = dict(n_channels=N_CHANNELS, n_samples=N_SAMPLES)

def build_models():
    return {
        'EEGNet':                EEGNet(**COMMON),
        'EEGNet_Inc2_SmallMed':  EEGNetInception(**COMMON, kernel_sizes=(16, 32)),
        'EEGNet_Inc2_MedLarge':  EEGNetInception(**COMMON, kernel_sizes=(32, 64)),
        'EEGNet_Inc3':           EEGNetInception(**COMMON, kernel_sizes=(16, 32, 64)),
        'EEGNet_Inc3_Narrow':    EEGNetInception(**COMMON, kernel_sizes=(8, 16, 32)),
        'EEGNet_Inc3_Wide':      EEGNetInception(**COMMON, kernel_sizes=(32, 64, 128)),
    }

_preview = build_models()
print(f'{"Model":<25} {"Params":>10}')
print('-' * 37)
for name, m in _preview.items():
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'{name:<25} {n:>10,}')
del _preview

## 6. Training & Evaluation Helpers

In [ ]:
def train(model, train_loader, val_loader, epochs=EPOCHS, lr=LR, verbose=True):
    """Train in place; return history dict with per-epoch train loss and val acc."""
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    history = {'train_loss': [], 'val_acc': []}

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for X, y in train_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        model.eval()
        correct = total = 0
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(DEVICE), y.to(DEVICE)
                correct += (model(X).argmax(1) == y).sum().item()
                total   += len(y)
        val_acc = correct / total

        history['train_loss'].append(total_loss)
        history['val_acc'].append(val_acc)
        if verbose:
            print(f'  Epoch {epoch:3d} | Loss: {total_loss:7.2f} | Val Acc: {val_acc:.4f}')

    return history


def evaluate(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            correct += (model(X).argmax(1) == y).sum().item()
            total   += len(y)
    return correct / total


def collect_predictions(model, loader):
    model.eval()
    all_probs, all_preds, all_targets = [], [], []
    with torch.no_grad():
        for X, y in loader:
            X = X.to(DEVICE)
            logits = model(X)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            preds = probs.argmax(axis=1)
            all_probs.append(probs)
            all_preds.append(preds)
            all_targets.append(y.numpy())
    return (
        np.concatenate(all_targets),
        np.concatenate(all_preds),
        np.concatenate(all_probs),
    )

## 7. Train All 6 Variants

Trains baseline EEGNet plus all five inception variants under identical
settings. Re-seeds per model so weight initialization and data shuffling
are matched across runs.

In [ ]:
models = build_models()
results = {}

for name, m in models.items():
    print(f'\n{"=" * 60}')
    print(f'Training: {name}')
    print('=' * 60)

    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    history = train(m, train_loader, val_loader, epochs=EPOCHS, lr=LR)

    split_results = {}
    for split_name, loader in [('Train', train_loader),
                               ('Validation', val_loader),
                               ('Test', test_loader)]:
        y_true, y_pred, y_prob = collect_predictions(m, loader)
        split_results[split_name] = {
            'y_true': y_true, 'y_pred': y_pred, 'y_prob': y_prob,
        }

    y_true = split_results['Test']['y_true']
    y_pred = split_results['Test']['y_pred']
    y_prob = split_results['Test']['y_prob'][:, 1]

    train_acc = accuracy_score(split_results['Train']['y_true'],
                               split_results['Train']['y_pred'])
    val_acc   = accuracy_score(split_results['Validation']['y_true'],
                               split_results['Validation']['y_pred'])
    test_acc  = accuracy_score(y_true, y_pred)
    auc       = roc_auc_score(y_true, y_prob)

    print(f'\n>>> {name}')
    print(f'    Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Test Acc: {test_acc:.4f}')
    print(f'    Test ROC AUC: {auc:.4f}')

    results[name] = {
        'model':         m,
        'history':       history,
        'split_results': split_results,
        'params':        sum(p.numel() for p in m.parameters() if p.requires_grad),
        'train_acc':     train_acc,
        'val_acc':       val_acc,
        'test_acc':      test_acc,
        'test_auc':      auc,
    }

## 8. Per-Model Performance Diagnostics

Confusion matrix, ROC, PR curve, probability histogram, and training curve
for every trained model.

In [ ]:
def plot_model_diagnostics(model_name, split_results, history):
    y_true = split_results['Test']['y_true']
    y_pred = split_results['Test']['y_pred']
    y_prob = split_results['Test']['y_prob'][:, 1]

    cm        = confusion_matrix(y_true, y_pred)
    fpr, tpr, _         = roc_curve(y_true, y_prob)
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)

    fig, axes = plt.subplots(2, 3, figsize=(17, 10))

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Left', 'Right'], yticklabels=['Left', 'Right'],
                ax=axes[0, 0])
    axes[0, 0].set_title(f'{model_name} — Confusion Matrix')
    axes[0, 0].set_xlabel('Predicted'); axes[0, 0].set_ylabel('True')

    split_names = list(split_results.keys())
    split_accs  = [accuracy_score(split_results[n]['y_true'],
                                  split_results[n]['y_pred']) for n in split_names]
    axes[0, 1].bar(split_names, split_accs, color=['#4c78a8', '#f58518', '#54a24b'])
    axes[0, 1].set_ylim(0.0, 1.0)
    axes[0, 1].set_title('Accuracy by Split')
    axes[0, 1].set_ylabel('Accuracy')
    for i, acc in enumerate(split_accs):
        axes[0, 1].text(i, acc + 0.02, f'{acc:.3f}', ha='center')

    axes[0, 2].plot(fpr, tpr, label=f'AUC = {auc:.3f}', color='#4c78a8')
    axes[0, 2].plot([0, 1], [0, 1], '--', color='gray')
    axes[0, 2].set_title('ROC Curve')
    axes[0, 2].set_xlabel('False Positive Rate')
    axes[0, 2].set_ylabel('True Positive Rate')
    axes[0, 2].legend(loc='lower right')

    axes[1, 0].hist(y_prob[y_true == 0], bins=20, alpha=0.65, label='True Left', color='#4c78a8')
    axes[1, 0].hist(y_prob[y_true == 1], bins=20, alpha=0.65, label='True Right', color='#f58518')
    axes[1, 0].axvline(0.5, linestyle='--', color='black', linewidth=1)
    axes[1, 0].set_title('Predicted P(Right) on Test')
    axes[1, 0].set_xlabel('P(Right)'); axes[1, 0].set_ylabel('Count')
    axes[1, 0].legend()

    axes[1, 1].plot(recall, precision, color='#54a24b')
    axes[1, 1].set_title('Precision–Recall Curve')
    axes[1, 1].set_xlabel('Recall'); axes[1, 1].set_ylabel('Precision')
    axes[1, 1].set_ylim(0.0, 1.05); axes[1, 1].set_xlim(0.0, 1.0)
    axes[1, 1].grid(alpha=0.3)

    epochs_range = np.arange(1, len(history['val_acc']) + 1)
    ax_loss = axes[1, 2]
    ax_loss.plot(epochs_range, history['train_loss'], color='#4c78a8', label='Train Loss')
    ax_loss.set_xlabel('Epoch')
    ax_loss.set_ylabel('Train Loss', color='#4c78a8')
    ax_loss.tick_params(axis='y', labelcolor='#4c78a8')
    ax_acc = ax_loss.twinx()
    ax_acc.plot(epochs_range, history['val_acc'], color='#f58518', label='Val Acc')
    ax_acc.set_ylabel('Val Acc', color='#f58518')
    ax_acc.tick_params(axis='y', labelcolor='#f58518')
    axes[1, 2].set_title('Training Curve')

    fig.suptitle(f'{model_name} — Diagnostics', fontsize=14)
    plt.tight_layout()
    plt.show()

    print(f'\n{model_name} classification report (Test):')
    print(classification_report(y_true, y_pred,
                                target_names=['Left hand', 'Right hand']))


for name, r in results.items():
    plot_model_diagnostics(name, r['split_results'], r['history'])

## 9. Cross-Model Comparison

In [ ]:
baseline_acc = results['EEGNet']['test_acc']

names      = list(results.keys())
test_accs  = [results[n]['test_acc'] for n in names]
aucs       = [results[n]['test_auc'] for n in names]
colors     = ['#888888'] + ['#4c78a8'] * 2 + ['#f58518'] * 3

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

bars = axes[0].bar(names, test_accs, color=colors, edgecolor='white')
axes[0].axhline(baseline_acc, color='#888888', linestyle='--',
                linewidth=1, label='Baseline')
for bar, acc in zip(bars, test_accs):
    axes[0].text(bar.get_x() + bar.get_width() / 2, acc + 0.003,
                 f'{acc:.3f}', ha='center', va='bottom', fontsize=9)
lo = max(0.0, min(test_accs) - 0.05)
hi = min(1.0, max(test_accs) + 0.05)
axes[0].set_ylim(lo, hi)
axes[0].set_ylabel('Test Accuracy')
axes[0].set_title('Test Accuracy: Baseline vs Inception Variants')
axes[0].set_xticklabels(names, rotation=20, ha='right', fontsize=9)
axes[0].legend()

bars = axes[1].bar(names, aucs, color=colors, edgecolor='white')
for bar, v in zip(bars, aucs):
    axes[1].text(bar.get_x() + bar.get_width() / 2, v + 0.003,
                 f'{v:.3f}', ha='center', va='bottom', fontsize=9)
lo = max(0.0, min(aucs) - 0.05)
hi = min(1.0, max(aucs) + 0.05)
axes[1].set_ylim(lo, hi)
axes[1].set_ylabel('Test ROC AUC')
axes[1].set_title('Test ROC AUC: Baseline vs Inception Variants')
axes[1].set_xticklabels(names, rotation=20, ha='right', fontsize=9)

plt.tight_layout()
plt.show()

# Validation accuracy curves
fig, ax = plt.subplots(figsize=(10, 5))
for name, r in results.items():
    ax.plot(np.arange(1, EPOCHS + 1), r['history']['val_acc'], label=name)
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Accuracy')
ax.set_title('Validation Accuracy over Training')
ax.legend(loc='lower right', fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print('FINAL SUMMARY')
print('=' * 82)
print(f'{"Model":<25} {"Params":>8} {"Train":>8} {"Val":>8} {"Test":>8} {"AUC":>8} {"Delta":>9}')
print('-' * 82)
for name, r in results.items():
    delta = r['test_acc'] - baseline_acc
    delta_str = '—' if name == 'EEGNet' else f'{delta:+.4f}'
    print(f'{name:<25} {r["params"]:>8,} {r["train_acc"]:>8.4f} '
          f'{r["val_acc"]:>8.4f} {r["test_acc"]:>8.4f} {r["test_auc"]:>8.4f} {delta_str:>9}')

csv_path = os.path.join(RESULTS_DIR, 'summary.csv')
with open(csv_path, 'w') as f:
    f.write('model,params,train_acc,val_acc,test_acc,test_auc,delta_vs_baseline\n')
    for name, r in results.items():
        delta = r['test_acc'] - baseline_acc
        f.write(f'{name},{r["params"]},{r["train_acc"]:.4f},'
                f'{r["val_acc"]:.4f},{r["test_acc"]:.4f},'
                f'{r["test_auc"]:.4f},{delta:+.4f}\n')
print(f'\nSaved summary → {csv_path}')